In [ ]:
import numpy as np

In [51]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [52]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [53]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


In [55]:
# 1 ## [База] 1. Выручка по чистым данным
#Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?
rev_sep = amount[clean & (month == "2026-09")].sum()
rev_oct = amount[clean & (month == "2026-10")].sum()

# Изменение выручки в процентах (от сентября к октябрю)
pct_change = ((rev_oct - rev_sep) / rev_sep) * 100

print("Строк строк осталось в `clean`:", clean.sum())
print(f"Выручка сентябрь: {rev_sep} ₸")
print(f"Выручка октябрь:  {rev_oct} ₸")
print(f"Изменение:        {pct_change}%")

Строк строк осталось в `clean`: 4739
Выручка сентябрь: 35243410.0 ₸
Выручка октябрь:  34419520.0 ₸
Изменение:        -2.337713632137185%


In [105]:
## [База] 2. Таблица «категория × месяц»
#Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
#Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

cats = np.unique(cat[clean])
t = np.array([[amount[ (cat == c) & (month == m)].sum() for m in ("2026-09", "2026-10")] for c in cats])
print("Выручка по месяцам (сентябрь, октябрь):", t.sum(axis=0))
for i in range(len(cats)):
    print(f"{cats[i]} | {t[i,0]} | {t[i,1]}")

pct = (t[:, 1] - t[:, 0]) / t[:, 0]
print("Топ по выручке:", cats[t.sum(axis=1).argmax()])
print("Сильнее упала:", cats[pct.argmin()], f"({pct.min():+.1%})")


Выручка по месяцам (сентябрь, октябрь): [60704470. 49178890.]
Дом | 5920970.0 | 5945700.0
Книги | 1253720.0 | 1303970.0
Красота | 3079510.0 | 2987950.0
Одежда | 10792720.0 | 9508740.0
Электроника | 39657550.0 | 29432530.0
Топ по выручке: Электроника
Сильнее упала: Электроника (-25.8%)


In [157]:
## [База] 3. Средний чек по категориям в октябре
#Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
#Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?
count = np.array([
    [((cat == c) & (month == m) & clean).sum()
     for m in ("2026-09", "2026-10")]
    for c in cats
])
print("Количество заказов:")
for i in range(len(cats)):
    print(f"{cats[i]} | {count[i,0]} | {count[i,1]}")
print("\nСредний чек:")
avg = t / count
for i in range(len(cats)):
    print(f"{cats[i]} | {avg[i,1]:.2f}")
 #print("Высокий чек:", cats.avg.argmax())
print("\nСамый высокий средний чек:", cats[avg[:,1].argmax()])


Количество заказов:
Дом | 476 | 476
Книги | 265 | 299
Красота | 422 | 435
Одежда | 678 | 652
Электроника | 531 | 505

Средний чек:
Дом | 12490.97
Книги | 4361.10
Красота | 6868.85
Одежда | 14583.96
Электроника | 58282.24

Самый высокий средний чек: Электроника


In [161]:
## [Стандарт] 4. Доля городов в выручке месяца
# Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
# Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?
cities = np.unique(city)
months = np.array(["2026-09", "2026-10"])

# Таблица: строка = город, столбец = месяц
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])
for c, r in zip(cities, table):
    print(f"{c:<10} {r[0]:>12,.0f} {r[1]:>12,.0f}")
print(table.shape)
shares = table / table.sum(axis=0)

print("\nДоля городов в выручке (Сентябрь | Октябрь):")
for c, r in zip(cities, shares):
    print(f"{c:<10} {r[0]:>11.2%} {r[1]:>11.2%}")

# Проверка, что каждый столбец в сумме даёт 1
print("\nПроверка суммы по столбцам:", shares.sum(axis=0))


Актобе        2,576,960    2,540,420
Алматы       14,501,070   14,643,020
Астана        9,034,220    8,850,530
Атырау          923,990      969,860
Караганда     3,648,520    3,371,680
Шымкент       4,558,650    4,044,010
(6, 2)

Доля городов в выручке (Сентябрь | Октябрь):
Актобе           7.31%       7.38%
Алматы          41.15%      42.54%
Астана          25.63%      25.71%
Атырау           2.62%       2.82%
Караганда       10.35%       9.80%
Шымкент         12.93%      11.75%

Проверка суммы по столбцам: [1. 1.]


In [169]:
## [Стандарт] 5. Кто дал падение
# Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
# Напишите менеджеру одну фразу: где искать причину.

city_diff = table[:, 1] - table[:, 0]

# Общее изменение выручки по всем городам
total_diff = city_diff.sum()

# Вклад каждого города в общее падение
contribution = city_diff / total_diff

print("Вклад городов в общее изменение выручки:")
for c, d, p in zip(cities, city_diff, contribution):
    print(f"{c:<10} | {d:>+11,.0f} ₸ | {p:>7.2%}")

# Вклад Шымкента
idx_shym = np.where(cities == "Шымкент")[0][0]
shym_contrib = contribution[idx_shym]
print(f"\nВклад Шымкента в общее падение: {shym_contrib:.2%}")


Вклад городов в общее изменение выручки:
Актобе     |     -36,540 ₸ |   4.44%
Алматы     |    +141,950 ₸ | -17.23%
Астана     |    -183,690 ₸ |  22.30%
Атырау     |     +45,870 ₸ |  -5.57%
Караганда  |    -276,840 ₸ |  33.60%
Шымкент    |    -514,640 ₸ |  62.46%

Вклад Шымкента в общее падение: 62.46%


In [185]:
## [Стандарт] 6. Исправьте код от AI
#AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
#```python
#share = table / table.sum(axis=1)
#```
## несовпаденее размерностей, выравниваем через keepdims
share = table / table.sum(axis=1, keepdims=True)
print(share)

[[0.50357019 0.49642981]
 [0.49756469 0.50243531]
 [0.50513538 0.49486462]
 [0.48788975 0.51211025]
 [0.51971739 0.48028261]
 [0.52991168 0.47008832]]


In [186]:
## [Челлендж] 7. Стандартизация суммы заказа
# Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
# Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.
## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
#Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
#Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
#Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.